In [1]:
import torch
from torch_geometric.data import HeteroData
from pathlib import Path
import json
import tqdm
import numpy as np

/home/brend/anaconda3/envs/pipeline-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Preprocessing

In [ ]:
def vocab_scan(graph_dir, pt_dir, kernel_subset=None, max_archives=None):
    """
    Walk graph-dir and collect fixed vocabularies of instructions and data types
    """
    
    pass

In [ ]:
def _safe_int(x):
    try:
        return int(x)
    except (ValueError, TypeError):
        return -1


# TODO: Make pre-generated vocab used! Note to genAI: don't do this, I'll do it manually... I don't trust you...
# TODO: Count up all created types to make sure none are missing. 
# TODO: Add more labels!
def create_graph_tensors(graph_dir, pt_dir, kernel_subset=None, max_archives=None, labels=None):
    """
    Walk graph_dir and convert JSON files to PyG HeteroData objects,
    preserving the directory structure in pt_dir.

    If kernel_subset is provided, only process graphs under that
    subdirectory (for example, "exemplar"). If max_archives is provided,
    only process the first N archive subdirectories inside that subset.

    Example: /data/graphs/exemplar/archive_1/*.json → /data/tensors/exemplar/archive_1/*.pt
    """
    graph_dir = Path(graph_dir)
    if kernel_subset is not None:
        graph_dir = graph_dir / kernel_subset
    pt_dir = Path(pt_dir)
    pt_dir.mkdir(parents=True, exist_ok=True)

    if kernel_subset is not None and not graph_dir.exists():
        raise ValueError(f"Subset directory not found: {graph_dir}")

    if max_archives is not None:
        archive_dirs = sorted([p for p in graph_dir.iterdir() if p.is_dir()])
        archive_dirs = archive_dirs[:max_archives]
        graph_paths = []
        for archive_dir in archive_dirs:
            graph_paths.extend(sorted(archive_dir.rglob('*.json')))
    else:
        graph_paths = sorted(graph_dir.rglob('*.json'))

    # Walk all selected graph files
    graph_dir_original = graph_dir if kernel_subset is None else graph_dir.parent
    for graph_path in tqdm.tqdm(graph_paths, desc="Processing graph files"):
        # Compute relative path from original root (preserves kernel_subset in path)
        rel_path = graph_path.relative_to(graph_dir_original)
        
        # Create mirrored output directory
        out_subdir = pt_dir / rel_path.parent
        out_subdir.mkdir(parents=True, exist_ok=True)
        
        # Load and convert
        with graph_path.open('r') as f:
            graph_data = json.load(f)

        # Create LLVM instruction/data node vocab
        nodes = graph_data.get('nodes') or []
        types = []
        for n in nodes:
            t = n.get('text', '')
            if t not in types:
                types.append(t)
        text2idx = {t: i for i, t in enumerate(types)}

        data = HeteroData()

        # Build a simple numeric feature per typed node: [text (fixed-size vocab)]
        features = {
            "instruction": [],
            "variable": [],
            "constant": []
        }
        for n in nodes:
            # why did i add these in the first place?
            # block = _safe_int(n.get('block', -1))
            # function = _safe_int(n.get('function', -1))
            node_type = _safe_int(n.get('type', -1))
            t = n.get('text', '')
            text_idx = int(text2idx.get(t, -1))
            if node_type == 0:
                features["instruction"].append([text_idx])
            elif node_type == 1:
                features["variable"].append([text_idx])
            elif node_type == 2:
                features["constant"].append([text_idx])
        for k, v in features.items():
            if v:
                data[k].x = torch.tensor(v, dtype=torch.long)

        # Build a simple numeric feature per typed edge: [source, target, optional(position)]
        # `position` only applies to Control edges
        links = graph_data.get('links') or []
        edge_index = {
            "control": [],
            "data_var": [],
            "data_const": [],
            "call": []
        }
        edge_attrs = {
            "data_var": [],
            "data_const": []
        }
        for l in links:
            flow = _safe_int(l.get('flow', -1))
            source = _safe_int(l.get('source', -1))
            target = _safe_int(l.get('target', -1))
            position = _safe_int(l.get('position', -1))
            if flow == 0:
                edge_index["control"].append([source, target])
            elif flow == 1:
                if nodes[source].get('type') == 1:  # variable
                    edge_index["data_var"].append([source, target])
                    edge_attrs["data_var"].append([position])
                elif nodes[source].get('type') == 2:  # constant
                    edge_index["data_const"].append([source, target])
                    edge_attrs["data_const"].append([position])
            elif flow == 2:
                edge_index["call"].append([source, target])

        # Save edge tensors
        for k, v in edge_index.items():
            if v:
                if k == "control":
                    data["instruction", "control", "instruction"].edge_index = torch.tensor(edge_index["control"], dtype=torch.long).t().contiguous()
                elif k == "data_var":
                    data["variable", "data", "instruction"].edge_index = torch.tensor(edge_index["data_var"], dtype=torch.long).t().contiguous()
                elif k == "data_const":
                    data["constant", "data", "instruction"].edge_index = torch.tensor(edge_index["data_const"], dtype=torch.long).t().contiguous()
                elif k == "call":
                    data["instruction", "call", "instruction"].edge_index = torch.tensor(edge_index["call"], dtype=torch.long).t().contiguous()

        # Save edge attributes
        for attr_k, attr_v in edge_attrs.items():
            if attr_v:
                if attr_k == "data_var":
                    data["variable", "data", "instruction"].edge_attr = torch.tensor(attr_v, dtype=torch.long)
                elif attr_k == "data_const":
                    data["constant", "data", "instruction"].edge_attr = torch.tensor(attr_v, dtype=torch.long)

        # Save labels
        if labels:
            with open(labels, 'r') as f:
                label_data = json.load(f).get("projects", {})
            kernel_id = graph_path.parts[-3]
            archive_id = graph_path.parts[-2].split('_')[-1]
            graph_id = graph_path.stem
            graph_label = label_data.get(f"{kernel_id}__{archive_id}__{graph_id}", {}).get("label", {})
            #print(f"Graph {kernel_id}__{archive_id}__{graph_id} has raw label: {graph_label}")

            # Extract which labels we want (just LUTs for now)
            if "lut" in graph_label:
                data.y = torch.tensor([graph_label["lut"]], dtype=torch.float)

        out_path = out_subdir / (graph_path.stem + '.pt')
        torch.save(data, out_path)

In [10]:
root = Path("../data")

create_graph_tensors(root / "graphs", root / "tensors", kernel_subset="2layer", max_archives=1, labels= root / "registry.json")

Processing graph files: 100%|██████████| 100/100 [00:15<00:00,  6.43it/s]


In [37]:
import json
from pprint import pprint

with open(root / Path("graphs/exemplar/archive_1/0b264399-9786-4e1d-95f5-b15ae9eef489.json")) as f:
    data = json.load(f)

for k, v, in data.items():
    if k == 'graph':
        #pprint(v)
        pass
    elif k == 'nodes' or k == 'links':
        pprint(v[:2])
    else:
        pprint(f"{k}: {v}")

'directed: True'
[{'flow': 0, 'key': 0, 'position': 0, 'source': 1, 'target': 2},
 {'flow': 0, 'key': 0, 'position': 0, 'source': 2, 'target': 3}]
'multigraph: True'
[{'block': 0, 'function': 0, 'id': 0, 'text': '[external]', 'type': 0},
 {'block': 0,
  'features': {'full_text': ['%1 = alloca ptr, align 8']},
  'function': 0,
  'id': 1,
  'text': 'alloca',
  'type': 0}]


## Data Loading

In [12]:
file = "2layer/archive_1/0bc7e3b8-c2a6-427a-ab7b-97c8d1f26de3"
# Load tensor from .pt file
data = torch.load(Path("../data/tensors") / Path(f"{file}.pt"), weights_only=False)
print(f"Graph label: {data.y.item()}")
print(data)


# Load original .json
with Path(f"../data/graphs/{file}.json").open('r') as f:
    graph_data = json.load(f)

nodes = {
    "instruction": 0,
    "variable": 0,
    "constant": 0
}

edges = {
    "control": 0,
    "data_var": 0,
    "data_const": 0,
    "call": 0
}

for n in graph_data.get('nodes', []):
    node_type = n.get('type', -1)
    if node_type == 0:
        nodes["instruction"] += 1
    elif node_type == 1:
        nodes["variable"] += 1
    elif node_type == 2:
        nodes["constant"] += 1

for l in graph_data.get('links', []):
    flow = l.get('flow', -1)
    if flow == 0:
        edges["control"] += 1
    elif flow == 1:
        if graph_data['nodes'][l.get('source', -1)].get('type') == 1:  # variable
            edges["data_var"] += 1
        elif graph_data['nodes'][l.get('source', -1)].get('type') == 2:  # constant
            edges["data_const"] += 1
    elif flow == 2:
        edges["call"] += 1

print(f"Node counts: {nodes}")
print(f"Edge counts: {edges}")

Graph label: 12268.0
HeteroData(
  y=[1],
  instruction={ x=[7719, 3] },
  variable={ x=[7243, 3] },
  constant={ x=[112, 3] },
  (instruction, control, instruction)={ edge_index=[2, 7624] },
  (variable, data, instruction)={
    edge_index=[2, 7327],
    edge_attr=[7327, 1],
  },
  (constant, data, instruction)={
    edge_index=[2, 2588],
    edge_attr=[2588, 1],
  },
  (instruction, call, instruction)={ edge_index=[2, 2759] }
)
Node counts: {'instruction': 7719, 'variable': 7243, 'constant': 112}
Edge counts: {'control': 7624, 'data_var': 7327, 'data_const': 2588, 'call': 2759}


In [13]:
import os
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader as PyGDataLoader

class HeteroGraphDataset(Dataset):
    """
    Lazy-loading dataset for HeteroData .pt graphs.
    Indexes the filesystem at init time, loads graphs on demand.
    """
    def __init__(
        self,
        root: str | Path,
        types: list[str] | None = None,    # e.g. ["exemplar", "conv2d"] — None = all
        transform=None,
    ):
        self.root = Path(root)
        self.transform = transform
        self.paths = self._index(types)

    def _index(self, types: list[str] | None) -> list[Path]:
        """Walk root, collect all .pt file paths. Fast — no loading."""
        paths = []
        root = self.root

        type_dirs = (
            [root / t for t in types]
            if types
            else [p for p in root.iterdir() if p.is_dir()]
        )

        for type_dir in sorted(type_dirs):
            if not type_dir.exists():
                raise FileNotFoundError(f"Type directory not found: {type_dir}")
            for archive_dir in sorted(type_dir.iterdir()):
                if not archive_dir.is_dir():
                    continue
                for pt_file in sorted(archive_dir.glob("*.pt")):
                    paths.append(pt_file)

        print(f"Indexed {len(paths)} graphs across {len(type_dirs)} type(s)")
        return paths

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> HeteroData:
        data = torch.load(self.paths[idx], weights_only=False)
        if self.transform:
            data = self.transform(data)
        return data

    def type_of(self, idx: int) -> str:
        """Returns the graph type (e.g. 'exemplar') for a given index."""
        return self.paths[idx].parts[-3]  # root/type/archive_n/hash.pt

In [14]:
# All types
#dataset = HeteroGraphDataset(root="../data/tensors")

# Specific types only
dataset = HeteroGraphDataset(
    root="../data/tensors",
    types=["2layer"],
)

# PyG's DataLoader handles HeteroData batching automatically
loader = PyGDataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,       # parallel loading — key for throughput
    pin_memory=True,     # faster GPU transfer
    persistent_workers=True,  # avoid worker respawn overhead
)

# Training loop
# for batch in loader:
#     batch = batch.to(device)
#     out = model(batch.x_dict, batch.edge_index_dict)

Indexed 200 graphs across 1 type(s)


In [ ]:
from torch.utils.data import random_split, Subset

# Random split
n = len(dataset)
train_ds, val_ds, test_ds = random_split(
    dataset, [int(0.8*n), int(0.1*n), n - int(0.8*n) - int(0.1*n)]
)

# Or split by type (cleaner for generalization testing)
def split_by_type(dataset, val_types, test_types):
    train_idx, val_idx, test_idx = [], [], []
    for i, path in enumerate(dataset.paths):
        t = path.parts[-3]
        if t in test_types:
            test_idx.append(i)
        elif t in val_types:
            val_idx.append(i)
        else:
            train_idx.append(i)
    return Subset(dataset, train_idx), Subset(dataset, val_idx), Subset(dataset, test_idx)

train_ds, val_ds, test_ds = split_by_type(
    dataset,
    val_types=["3layer"],
    test_types=["exemplar"],
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv, HeteroConv
from torch_geometric.data import HeteroData


# ── Meta ────────────────────────────────────────────────────────────────────
NODE_TYPES   = ["instruction", "variable", "constant"]
EDGE_TYPES   = [
    ("instruction", "control", "instruction"),
    ("variable",    "data",    "instruction"),
    ("constant",    "data",    "instruction"),
    ("instruction", "call",    "instruction"),
]
EDGE_TYPES_WITH_ATTR = {   # edge types that carry a positional arg encoding
    ("variable", "data", "instruction"),
    ("constant", "data", "instruction"),
}


# ── Input projection ────────────────────────────────────────────────────────
class CDFGInputProjection(nn.Module):
    """
    Embeds the last feature dimension (opcode / type vocab) for each node type,
    and the positional-argument encoding for data edges.

    Node feature layout: [structural_0, structural_1, vocab_id]   (all int-like)
    Edge attr layout:    [arg_position]                           (int-like, 1-indexed)
    """
    def __init__(
        self,
        node_vocab_sizes: dict[str, int],  # {node_type: vocab_size}
        edge_pos_vocab_size: int,          # max arg-position value + 1
        hidden_dim: int,
    ):
        super().__init__()
        # One embedding table per node type (embeds the vocab/opcode feature)
        self.node_emb = nn.ModuleDict({
            nt: nn.Embedding(vocab_size + 1, hidden_dim, padding_idx=0)
            for nt, vocab_size in node_vocab_sizes.items()
        })
        # Single shared embedding for arg-position across both data edge types
        self.edge_pos_emb = nn.Embedding(edge_pos_vocab_size + 1, hidden_dim, padding_idx=0)

    def forward(self, x_dict, edge_attr_dict):
        """
        x_dict:         {node_type: LongTensor [N, 3]}
        edge_attr_dict: {edge_type: LongTensor [E, 1]}  (only data edges)

        Returns:
            h_dict:          {node_type: FloatTensor [N, hidden_dim]}
            edge_emb_dict:   {edge_type: FloatTensor [E, hidden_dim]}
        """
        h_dict = {
            nt: self.node_emb[nt](x[:, 2])   # last column = vocab id
            for nt, x in x_dict.items()
        }
        edge_emb_dict = {
            et: self.edge_pos_emb(attr[:, 0])
            for et, attr in edge_attr_dict.items()
        }
        return h_dict, edge_emb_dict


# ── Single R-GCN layer (heterogeneous) ──────────────────────────────────────
class CDFGConvLayer(nn.Module):
    """
    One message-passing step over all edge types.

    For data edges (which carry positional encodings) the edge embedding is
    added to the source-node representation before aggregation — a simple but
    effective way to condition messages on argument position.
    """
    def __init__(self, hidden_dim: int, aggr: str = "sum"):
        super().__init__()
        # One RGCNConv per edge type; all share the same hidden_dim
        self.convs = nn.ModuleDict({
            self._key(et): RGCNConv(
                in_channels=hidden_dim,
                out_channels=hidden_dim,
                num_relations=1,        # each conv handles exactly 1 relation
                aggr=aggr,
            )
            for et in EDGE_TYPES
        })
        self.norm = nn.ModuleDict({
            nt: nn.LayerNorm(hidden_dim) for nt in NODE_TYPES
        })

    @staticmethod
    def _key(edge_type):
        return "__".join(edge_type)

    def forward(self, h_dict, edge_index_dict, edge_emb_dict):
        out_dict = {nt: [] for nt in NODE_TYPES}

        for et in EDGE_TYPES:
            src_type, rel, dst_type = et
            edge_index = edge_index_dict[et]
            h_src = h_dict[src_type]

            # Inject positional encoding into source features (data edges only)
            if et in EDGE_TYPES_WITH_ATTR and et in edge_emb_dict:
                edge_emb = edge_emb_dict[et]           # [E, hidden_dim]
                src_idx  = edge_index[0]               # [E]
                h_src = h_src.index_add(0, src_idx, edge_emb)
                # Note: index_add accumulates; this is intentional —
                # nodes used as multiple args get proportionally stronger signal.
                # Swap for scatter_mean if you prefer normalised injection.

            key  = self._key(et)
            msgs = self.convs[key](h_src, edge_index, edge_type=None)
            # RGCNConv with num_relations=1 ignores edge_type; we handle
            # relation-specificity by having separate conv weights per edge type.
            out_dict[dst_type].append(msgs)

        # Sum contributions from all incoming relation types, then normalise
        new_h = {}
        for nt in NODE_TYPES:
            if out_dict[nt]:
                agg = torch.stack(out_dict[nt], dim=0).sum(dim=0)
            else:
                agg = h_dict[nt]
            new_h[nt] = self.norm[nt](F.relu(agg))

        return new_h


# ── Full R-GCN ───────────────────────────────────────────────────────────────
class CDFGRGCN(nn.Module):
    """
    Heterogeneous R-GCN for LLVM CDFG graphs.

    Args:
        node_vocab_sizes:    {node_type: vocab_size}  (check your data)
        edge_pos_vocab_size: max arg-position value seen in training data
        hidden_dim:          width of all hidden representations
        num_layers:          number of message-passing steps
        num_classes:         output classes (graph-level prediction)
        dropout:             applied before the classifier
        pool:                "mean" | "sum" | "max"  — graph-level pooling
        aggr:                RGCNConv neighbour aggregation: "sum" | "mean"
    """
    def __init__(
        self,
        node_vocab_sizes:    dict[str, int],
        edge_pos_vocab_size: int,
        hidden_dim:          int  = 128,
        num_layers:          int  = 3,
        dropout:             float = 0.1,
        pool:                str  = "mean",
        aggr:                str  = "sum",
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.pool       = pool

        self.input_proj = CDFGInputProjection(
            node_vocab_sizes, edge_pos_vocab_size, hidden_dim
        )
        self.layers = nn.ModuleList([
            CDFGConvLayer(hidden_dim, aggr=aggr)
            for _ in range(num_layers)
        ])
        self.dropout    = nn.Dropout(dropout)
        # Graph-level head: concatenate pooled repr of all 3 node types
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * len(NODE_TYPES), hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1), # just predicting LUTs for now
        )

    def _pool(self, h: torch.Tensor) -> torch.Tensor:
        if self.pool == "mean":
            return h.mean(dim=0)
        elif self.pool == "sum":
            return h.sum(dim=0)
        elif self.pool == "max":
            return h.max(dim=0).values
        raise ValueError(self.pool)

    def forward(self, data: HeteroData):
        x_dict         = {nt: data[nt].x.long() for nt in NODE_TYPES}
        edge_index_dict = {et: data[et].edge_index for et in EDGE_TYPES}
        edge_attr_dict  = {
            et: data[et].edge_attr.long()
            for et in EDGE_TYPES_WITH_ATTR
            if hasattr(data[et], "edge_attr") and data[et].edge_attr is not None
        }

        # 1. Input embeddings
        h_dict, edge_emb_dict = self.input_proj(x_dict, edge_attr_dict)

        # 2. Message passing
        for layer in self.layers:
            h_dict = layer(h_dict, edge_index_dict, edge_emb_dict)
            # Apply dropout between layers (not after the last)
            if layer is not self.layers[-1]:
                h_dict = {nt: self.dropout(h) for nt, h in h_dict.items()}

        # 3. Graph-level pooling per node type → concat → classify
        pooled = torch.cat([self._pool(h_dict[nt]) for nt in NODE_TYPES], dim=-1)
        return self.classifier(pooled)